# Team Features

Compile all feature engineering into a model-ready dataframe. 

In [1]:
SEASON = 2025

### Previous Tournament Results

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.read_parquet(r'..\data\preprocessed\womens_kaggle\tournament_results.parquet')

df = df.loc[(~df['Season'].isin([2020])) & (df['Season'] < SEASON), :].reset_index(drop=True)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results
0,2012,3101,Abilene Chr,-1.0,-1.0
1,2012,3102,Air Force,-1.0,-1.0
2,2012,3103,Akron,-1.0,-1.0
3,2012,3104,Alabama,-1.0,-1.0
4,2012,3105,Alabama A&M,-1.0,-1.0
...,...,...,...,...,...
4531,2024,3476,Stonehill,-1.0,-1.0
4532,2024,3477,East Texas A&M,-1.0,-1.0
4533,2024,3478,Le Moyne,-1.0,-1.0
4534,2024,3479,Mercyhurst,-1.0,-1.0


### Barttorvik Ratings

Omitted

In [3]:
# df_barttorvik = pd.read_parquet(r'..\data\preprocessed\womens_barttorvik\barttorvik.parquet')

# df_barttorvik

In [4]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\WTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

df_spellings.loc[df_spellings.shape[0]] = ['fdu', 3192]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,3394
1,a&m-corpus christi,3394
2,abilene chr,3101
3,abilene christian,3101
4,abilene-christian,3101
...,...,...
1171,youngstown st.,3464
1172,youngstown state,3464
1173,youngstown-st,3464
1174,youngstown-state,3464


In [5]:
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

len(spelling_to_id)

1170

In [6]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

def match_names(team_spellings, new_data_teams):
    df_match = pd.DataFrame(
        [
            [
                new_data_team,
                *process.extract(
                    new_data_team,
                    team_spellings,
                    scorer=token_sort_ratio,
                    limit=1
                )[0][:2]
            ] for new_data_team in tqdm(new_data_teams)
        ],
        columns=['New Data Team', 'Team Spelling', 'Match Score']
    ).sort_values('Match Score', ignore_index=True)

    team_to_spelling = dict(zip(df_match['New Data Team'], df_match['Team Spelling']))

    return df_match, team_to_spelling

C:\Users\mhugh\AppData\Local\Temp\ipykernel_15412\2578028529.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [7]:
# df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik['TEAM'].unique())

# df_match.head(25)

In [8]:
# df_barttorvik.insert(1, 'TeamID', df_barttorvik['TEAM'].map(team_to_spelling).map(spelling_to_id))

# df_barttorvik

In [9]:
# df_barttorvik.loc[df_barttorvik['TeamID'].isna(), :]

In [10]:
# df = pd.merge(
#     df,
#     df_barttorvik.drop(columns=['TEAM']),
#     how='left',
#     on=['Season', 'TeamID']
# )

# df

In [11]:
# df.loc[df['WIN%'].isna(), :]

### Past Seasons

In [12]:
df_ps = pd.read_parquet(r'..\data\preprocessed\womens_past_seasons\past_seasons_ratings.parquet')

df_ps

,Season,Team,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2012,Abilene Christian,NaN,NaN
1,2012,Air Force,-0.116981,-0.187368
2,2012,Akron,0.024465,-0.035480
3,2012,Alabama,0.052840,0.011622
4,2012,Alabama A&M,-0.141989,-0.161826
...,...,...,...,...
5077,2025,Wright State,-0.047642,-0.068898
5078,2025,Wyoming,0.071691,0.079227
5079,2025,Xavier,-0.205147,-0.105021
5080,2025,Yale,-0.099924,-0.042498


In [13]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_ps['Team'].unique())

df_match.head(25)

  0%|          | 0/363 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Abilene Christian,abilene christian,100
4,Quinnipiac,quinnipiac,100
5,Queens (NC),queens (nc),100
6,Purdue Fort Wayne,purdue fort wayne,100
7,Purdue,purdue,100
8,Providence,providence,100
9,Princeton,princeton,100


In [14]:
df_ps.insert(1, 'TeamID', df_ps['Team'].map(team_to_spelling).map(spelling_to_id))

df_ps

,Season,TeamID,Team,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2012,3101,Abilene Christian,NaN,NaN
1,2012,3102,Air Force,-0.116981,-0.187368
2,2012,3103,Akron,0.024465,-0.035480
3,2012,3104,Alabama,0.052840,0.011622
4,2012,3105,Alabama A&M,-0.141989,-0.161826
...,...,...,...,...,...
5077,2025,3460,Wright State,-0.047642,-0.068898
5078,2025,3461,Wyoming,0.071691,0.079227
5079,2025,3462,Xavier,-0.205147,-0.105021
5080,2025,3463,Yale,-0.099924,-0.042498


In [15]:
df = pd.merge(
    df,
    df_ps.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2012,3101,Abilene Chr,-1.0,-1.0,NaN,NaN
1,2012,3102,Air Force,-1.0,-1.0,-0.116981,-0.187368
2,2012,3103,Akron,-1.0,-1.0,0.024465,-0.035480
3,2012,3104,Alabama,-1.0,-1.0,0.052840,0.011622
4,2012,3105,Alabama A&M,-1.0,-1.0,-0.141989,-0.161826
...,...,...,...,...,...,...,...
4531,2024,3476,Stonehill,-1.0,-1.0,-0.234824,NaN
4532,2024,3477,East Texas A&M,-1.0,-1.0,-0.116192,NaN
4533,2024,3478,Le Moyne,-1.0,-1.0,NaN,NaN
4534,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN


In [16]:
df.loc[df['Past Year Efficiency Margin'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2012,3101,Abilene Chr,-1.0,-1.0,NaN,NaN
8,2012,3109,Alliant Intl,-1.0,-1.0,NaN,NaN
17,2012,3118,Armstrong St,-1.0,-1.0,NaN,NaN
20,2012,3121,Augusta,-1.0,-1.0,NaN,NaN
27,2012,3128,Birmingham So,-1.0,-1.0,NaN,NaN
...,...,...,...,...,...,...,...
4500,2024,3445,W Salem St,-1.0,-1.0,NaN,NaN
4501,2024,3446,W Texas A&M,-1.0,-1.0,NaN,NaN
4533,2024,3478,Le Moyne,-1.0,-1.0,NaN,NaN
4534,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN


In [17]:
df.loc[df['Past Year Efficiency Margin'].isna() & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
3427,2022,3126,Bethune-Cookman,-1.0,-0.666667,NaN,-0.058660
3469,2022,3169,CS Northridge,-1.0,-0.666667,NaN,-0.021810
3643,2022,3343,Princeton,-1.0,-0.333333,NaN,0.236506


### My Rankings

In [18]:
df_rankings = pd.concat(
    (
        pd.read_parquet(fr'..\data\preprocessed\womens_my_rankings\my_rankings_{season}.parquet')
        .assign(Season=season)
        for season in range(2012, SEASON) if season != 2020
    ),
    ignore_index=True
)

df_rankings.insert(0, 'Season', df_rankings.pop('Season'))

df_rankings.drop(columns=['Strength'], inplace=True)

df_rankings

,Season,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,Baylor,5.386146,0.545160,1.179988,0.634828,71.499272
1,2012,Stanford,4.561610,0.448443,1.148626,0.700183,69.596675
2,2012,Connecticut,4.445923,0.612579,1.178134,0.565556,69.690102
3,2012,Notre Dame,4.369786,0.536784,1.160459,0.623675,73.581011
4,2012,Delaware,3.802397,0.287090,1.059115,0.772024,67.789724
...,...,...,...,...,...,...,...
4197,2024,South Carolina State,-3.170798,-0.377634,0.685217,1.062851,69.006102
4198,2024,Wagner,-3.192290,-0.354522,0.689296,1.043818,71.754876
4199,2024,Long Island University,-3.272119,-0.320304,0.746029,1.066333,71.291510
4200,2024,Stonehill,-3.305148,-0.371073,0.696827,1.067900,70.574844


In [19]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_rankings['Team'].unique())

df_match.head(25)

  0%|          | 0/363 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Radford,radford,100
4,Maryland-Eastern Shore,maryland eastern shore,100
5,Alabama A&M,alabama a&m,100
6,Valparaiso,valparaiso,100
7,Evansville,evansville,100
8,Binghamton,binghamton,100
9,FDU,fdu,100


In [20]:
df_rankings.insert(1, 'TeamID', df_rankings['Team'].map(team_to_spelling).map(spelling_to_id))

df_rankings

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,3124,Baylor,5.386146,0.545160,1.179988,0.634828,71.499272
1,2012,3390,Stanford,4.561610,0.448443,1.148626,0.700183,69.596675
2,2012,3163,Connecticut,4.445923,0.612579,1.178134,0.565556,69.690102
3,2012,3323,Notre Dame,4.369786,0.536784,1.160459,0.623675,73.581011
4,2012,3174,Delaware,3.802397,0.287090,1.059115,0.772024,67.789724
...,...,...,...,...,...,...,...,...
4197,2024,3354,South Carolina State,-3.170798,-0.377634,0.685217,1.062851,69.006102
4198,2024,3447,Wagner,-3.192290,-0.354522,0.689296,1.043818,71.754876
4199,2024,3254,Long Island University,-3.272119,-0.320304,0.746029,1.066333,71.291510
4200,2024,3476,Stonehill,-3.305148,-0.371073,0.696827,1.067900,70.574844


In [21]:
df_rankings.loc[df_rankings['TeamID'].isna(), :]

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo


In [22]:
df = pd.merge(
    df,
    df_rankings.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,3101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,3102,Air Force,-1.0,-1.0,-0.116981,-0.187368,-1.427882,-0.223965,0.730399,0.954365,74.182539
2,2012,3103,Akron,-1.0,-1.0,0.024465,-0.035480,-0.379236,-0.008589,0.931008,0.939598,76.954858
3,2012,3104,Alabama,-1.0,-1.0,0.052840,0.011622,0.357797,0.012787,0.840928,0.828141,74.559256
4,2012,3105,Alabama A&M,-1.0,-1.0,-0.141989,-0.161826,-0.961452,-0.080677,0.828443,0.909120,68.735159
...,...,...,...,...,...,...,...,...,...,...,...,...
4531,2024,3476,Stonehill,-1.0,-1.0,-0.234824,NaN,-3.305148,-0.371073,0.696827,1.067900,70.574844
4532,2024,3477,East Texas A&M,-1.0,-1.0,-0.116192,NaN,-0.767316,-0.136922,0.862589,0.999511,77.283300
4533,2024,3478,Le Moyne,-1.0,-1.0,NaN,NaN,-0.988127,-0.134376,0.817137,0.951513,67.530652
4534,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
df.loc[df['Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2012,3101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
4488,2024,3432,Utica,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4500,2024,3445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4501,2024,3446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4534,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
df.loc[(df['Rating'].isna()) & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
3049,2021,3126,Bethune-Cookman,NaN,-0.666667,-0.004708,-0.057068,NaN,NaN,NaN,NaN,NaN
3091,2021,3169,CS Northridge,NaN,-0.666667,-0.091550,-0.015998,NaN,NaN,NaN,NaN,NaN
3257,2021,3335,Penn,NaN,-0.666667,0.153759,0.153866,NaN,NaN,NaN,NaN,NaN
3265,2021,3343,Princeton,NaN,-0.333333,0.342792,0.202387,NaN,NaN,NaN,NaN,NaN


### Starters

Omitted

In [25]:
# df_starters = pd.concat(
#     (
#         pd.read_parquet(fr'..\data\preprocessed\womens_starters\starters_{season}.parquet')
#         .assign(Season=season)
#         for season in range(2012, SEASON) if season != 2020
#     ),
#     ignore_index=True
# )

# df_starters.insert(0, 'Season', df_starters.pop('Season'))

# df_starters.rename(columns={'Rating': 'Starters'}, inplace=True)

# df_starters

In [26]:
# df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_starters['Team'].unique())

# df_match.head(25)

In [27]:
# df_starters.insert(1, 'TeamID', df_starters['Team'].map(team_to_spelling).map(spelling_to_id))

# df_starters

In [28]:
# df_starters.loc[df_starters['TeamID'].isna(), :]

In [29]:
# df = pd.merge(
#     df,
#     df_starters.drop(columns=['Team']),
#     how='left',
#     on=['Season', 'TeamID']
# )

# df

In [30]:
# df.loc[df['Starters'].isna(), :]

In [31]:
# df.loc[(df['Starters'].isna()) & (df['Past 4 Years Tournament Results'] > -1.0), :]

### Openskill Ratings

In [32]:
df_os = pd.concat(
    (
        pd.read_parquet(fr'..\data\preprocessed\womens_os_rankings\os_rankings_{season}.parquet')
        .assign(Season=season)
        for season in range(2012, SEASON) if season != 2020
    ),
    ignore_index=True
)

df_os.insert(0, 'Season', df_os.pop('Season'))

df_os.drop(columns=['Sigma'], inplace=True)

df_os

,Season,Team,Mu,OS Rating
0,2012,Baylor,59.530745,46.389680
1,2012,Stanford,54.907487,41.665374
2,2012,Connecticut,53.834791,41.593858
3,2012,Notre Dame,54.117201,41.353639
4,2012,Delaware,52.593131,38.213204
...,...,...,...,...
4197,2024,Alabama State,1.718860,-12.127996
4198,2024,McNeese State,2.277615,-12.622216
4199,2024,Houston Christian,1.520705,-12.900479
4200,2024,South Carolina State,-0.539184,-14.171589


In [33]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_os['Team'].unique())

df_match.head(25)

  0%|          | 0/363 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Baylor,baylor,100
4,Western Kentucky,western kentucky,100
5,North Dakota State,north dakota state,100
6,Alcorn State,alcorn state,100
7,North Florida,north florida,100
8,Alabama State,alabama state,100
9,Kennesaw State,kennesaw state,100


In [34]:
df_os.insert(1, 'TeamID', df_os['Team'].map(team_to_spelling).map(spelling_to_id))

df_os

,Season,TeamID,Team,Mu,OS Rating
0,2012,3124,Baylor,59.530745,46.389680
1,2012,3390,Stanford,54.907487,41.665374
2,2012,3163,Connecticut,53.834791,41.593858
3,2012,3323,Notre Dame,54.117201,41.353639
4,2012,3174,Delaware,52.593131,38.213204
...,...,...,...,...,...
4197,2024,3106,Alabama State,1.718860,-12.127996
4198,2024,3270,McNeese State,2.277615,-12.622216
4199,2024,3223,Houston Christian,1.520705,-12.900479
4200,2024,3354,South Carolina State,-0.539184,-14.171589


In [35]:
df = pd.merge(
    df,
    df_os.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating
0,2012,3101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,3102,Air Force,-1.0,-1.0,-0.116981,-0.187368,-1.427882,-0.223965,0.730399,0.954365,74.182539,8.791048,-4.497282
2,2012,3103,Akron,-1.0,-1.0,0.024465,-0.035480,-0.379236,-0.008589,0.931008,0.939598,76.954858,21.578057,9.007479
3,2012,3104,Alabama,-1.0,-1.0,0.052840,0.011622,0.357797,0.012787,0.840928,0.828141,74.559256,23.889403,10.898402
4,2012,3105,Alabama A&M,-1.0,-1.0,-0.141989,-0.161826,-0.961452,-0.080677,0.828443,0.909120,68.735159,22.077096,9.385125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4531,2024,3476,Stonehill,-1.0,-1.0,-0.234824,NaN,-3.305148,-0.371073,0.696827,1.067900,70.574844,2.006163,-10.928439
4532,2024,3477,East Texas A&M,-1.0,-1.0,-0.116192,NaN,-0.767316,-0.136922,0.862589,0.999511,77.283300,20.249666,7.487434
4533,2024,3478,Le Moyne,-1.0,-1.0,NaN,NaN,-0.988127,-0.134376,0.817137,0.951513,67.530652,23.057138,10.392918
4534,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
df.loc[df['OS Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating
0,2012,3101,Abilene Chr,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2012,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2012,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2012,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2012,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4488,2024,3432,Utica,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4500,2024,3445,W Salem St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4501,2024,3446,W Texas A&M,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4534,2024,3479,Mercyhurst,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Standard Stats

Omitted

In [37]:
# df_ss = pd.read_parquet('../data/preprocessed/womens_standard_stats/standard_stats.parquet')

# df_ss

In [38]:
# df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_ss['Team'].unique())

# df_match.head(25)

In [39]:
# df_ss.insert(1, 'TeamID', df_ss['Team'].map(team_to_spelling).map(spelling_to_id))

# df_ss

In [40]:
# df = pd.merge(
#     df,
#     df_ss.drop(columns=['Team']),
#     how='left',
#     on=['Season', 'TeamID']
# )

# df

In [41]:
# df.loc[df['Team Win%'].isna(), :]

### Map to Matchups

In [42]:
df_mod = pd.read_csv(r'..\data\unprocessed\kaggle\WNCAATourneyDetailedResults.csv')[['Season', 'DayNum', 'WTeamID', 'LTeamID']]

df_mod = df_mod.loc[df_mod['Season'].between(2012, SEASON, inclusive='left'), :].reset_index(drop=True)

df_mod

,Season,DayNum,WTeamID,LTeamID
0,2012,138,3116,3173
1,2012,138,3163,3341
2,2012,138,3177,3140
3,2012,138,3211,3353
4,2012,138,3243,3343
...,...,...,...,...
763,2024,147,3163,3425
764,2024,147,3234,3261
765,2024,151,3234,3163
766,2024,151,3376,3301


Get seeding

In [43]:
df_seeds = pd.read_csv(r'..\data\unprocessed\kaggle\WNCAATourneySeeds.csv')

df_seeds = df_seeds.loc[df_seeds['Season'] >= 2012, :].reset_index(drop=True)

df_seeds.insert(2, 'Play In', df_seeds['Seed'].str.endswith(('a', 'b')))
df_seeds.insert(2, 'Region', df_seeds['Seed'].str[0])
df_seeds['Seed'] = df_seeds['Seed'].str.extract('(\d+)').astype(int)

df_seeds.insert(1, 'Region Seed', df_seeds['Region'] + df_seeds['Seed'].astype(str).str.zfill(2))

df_seeds

,Season,Region Seed,Seed,Region,Play In,TeamID
0,2012,W01,1,W,False,3124
1,2012,W02,2,W,False,3397
2,2012,W03,3,W,False,3174
3,2012,W04,4,W,False,3210
4,2012,W05,5,W,False,3207
...,...,...,...,...,...,...
775,2024,Z12,12,Z,True,3435
776,2024,Z13,13,Z,False,3267
777,2024,Z14,14,Z,False,3238
778,2024,Z15,15,Z,False,3263


Remove play-ins

In [44]:
df_mod = pd.merge(
    df_mod,
    df_seeds[['Season', 'Region', 'Seed', 'Play In', 'TeamID']]
    .rename(columns={'TeamID': 'WTeamID', 'Region': 'WTeamRegion', 'Seed': 'WTeamSeed', 'Play In': 'WTeamPlayIn'}),
    how='left',
    on=['Season', 'WTeamID'],
)

df_mod

,Season,DayNum,WTeamID,LTeamID,WTeamRegion,WTeamSeed,WTeamPlayIn
0,2012,138,3116,3173,Z,6,False
1,2012,138,3163,3341,Y,1,False
2,2012,138,3177,3140,W,7,False
3,2012,138,3211,3353,Y,11,False
4,2012,138,3243,3343,Y,8,False
...,...,...,...,...,...,...,...
763,2024,147,3163,3425,Z,3,False
764,2024,147,3234,3261,Y,1,False
765,2024,151,3234,3163,Y,1,False
766,2024,151,3376,3301,W,1,False


In [45]:
df_mod = pd.merge(
    df_mod,
    df_seeds[['Season', 'Region', 'Seed', 'Play In', 'TeamID']]
    .rename(columns={'TeamID': 'LTeamID', 'Region': 'LTeamRegion', 'Seed': 'LTeamSeed', 'Play In': 'LTeamPlayIn'}),
    how='left',
    on=['Season', 'LTeamID'],
)

df_mod

,Season,DayNum,WTeamID,LTeamID,WTeamRegion,WTeamSeed,WTeamPlayIn,LTeamRegion,LTeamSeed,LTeamPlayIn
0,2012,138,3116,3173,Z,6,False,Z,11,False
1,2012,138,3163,3341,Y,1,False,Y,16,False
2,2012,138,3177,3140,W,7,False,W,10,False
3,2012,138,3211,3353,Y,11,False,Y,6,False
4,2012,138,3243,3343,Y,8,False,Y,9,False
...,...,...,...,...,...,...,...,...,...,...
763,2024,147,3163,3425,Z,3,False,Z,1,False
764,2024,147,3234,3261,Y,1,False,Y,3,False
765,2024,151,3234,3163,Y,1,False,Z,3,False
766,2024,151,3376,3301,W,1,False,X,3,False


In [46]:
df_mod = df_mod.loc[
    ~(  # exclude if teams are in the same region and both play-ins
        (df_mod['WTeamRegion'] == df_mod['LTeamRegion']) & 
        (df_mod['WTeamPlayIn']) & 
        (df_mod['LTeamPlayIn'])
    ), 
    :
].reset_index(drop=True)

df_mod

,Season,DayNum,WTeamID,LTeamID,WTeamRegion,WTeamSeed,WTeamPlayIn,LTeamRegion,LTeamSeed,LTeamPlayIn
0,2012,138,3116,3173,Z,6,False,Z,11,False
1,2012,138,3163,3341,Y,1,False,Y,16,False
2,2012,138,3177,3140,W,7,False,W,10,False
3,2012,138,3211,3353,Y,11,False,Y,6,False
4,2012,138,3243,3343,Y,8,False,Y,9,False
...,...,...,...,...,...,...,...,...,...,...
751,2024,147,3163,3425,Z,3,False,Z,1,False
752,2024,147,3234,3261,Y,1,False,Y,3,False
753,2024,151,3234,3163,Y,1,False,Z,3,False
754,2024,151,3376,3301,W,1,False,X,3,False


Remap to Team A / Team B format

In [47]:
df_mod = pd.DataFrame({
    'Season': list(df_mod['Season'])*2,
    'Result': [1 for _ in range(df_mod.shape[0])] + [-1 for _ in range(df_mod.shape[0])],
    'Team A ID': list(df_mod['WTeamID']) + list(df_mod['LTeamID']),
    'Team B ID': list(df_mod['LTeamID']) + list(df_mod['WTeamID']),
    'Team A Region': list(df_mod['WTeamRegion']) + list(df_mod['LTeamRegion']),
    'Team B Region': list(df_mod['LTeamRegion']) + list(df_mod['WTeamRegion']),
    'Team A Seed': list(df_mod['WTeamSeed']) + list(df_mod['LTeamSeed']),
    'Team B Seed': list(df_mod['LTeamSeed']) + list(df_mod['WTeamSeed']),
})

df_mod

,Season,Result,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed
0,2012,1,3116,3173,Z,Z,6,11
1,2012,1,3163,3341,Y,Y,1,16
2,2012,1,3177,3140,W,W,7,10
3,2012,1,3211,3353,Y,Y,11,6
4,2012,1,3243,3343,Y,Y,8,9
...,...,...,...,...,...,...,...,...
1507,2024,-1,3425,3163,Z,Z,1,3
1508,2024,-1,3261,3234,Y,Y,3,1
1509,2024,-1,3163,3234,Z,Y,3,1
1510,2024,-1,3301,3376,X,W,3,1


Get round of matchup

In [48]:
same_region = df_mod['Team A Region'] == df_mod['Team B Region']

# round_0_condition = (df_mod['team0_playin'] == 1) & (df_mod['team1_playin'] == 1)  # no play-in games in this data

round_1_condition = df_mod['Team A Seed'] + df_mod['Team B Seed'] == 17

round_2_condition = (
    (df_mod['Team A Seed'].isin([1, 16]) & df_mod['Team B Seed'].isin([8, 9])) | 
    (df_mod['Team A Seed'].isin([8, 9]) & df_mod['Team B Seed'].isin([1, 16])) |
    (df_mod['Team A Seed'].isin([5, 12]) & df_mod['Team B Seed'].isin([4, 13])) | 
    (df_mod['Team A Seed'].isin([4, 13]) & df_mod['Team B Seed'].isin([5, 12])) |
    (df_mod['Team A Seed'].isin([6, 11]) & df_mod['Team B Seed'].isin([3, 14])) | 
    (df_mod['Team A Seed'].isin([3, 14]) & df_mod['Team B Seed'].isin([6, 11])) |
    (df_mod['Team A Seed'].isin([7, 10]) & df_mod['Team B Seed'].isin([2, 15])) | 
    (df_mod['Team A Seed'].isin([2, 15]) & df_mod['Team B Seed'].isin([7, 10]))
)

round_3_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9]) & df_mod['Team B Seed'].isin([5, 12, 4, 13])) | 
    (df_mod['Team A Seed'].isin([5, 12, 4, 13]) & df_mod['Team B Seed'].isin([1, 16, 8, 9])) |
    (df_mod['Team A Seed'].isin([6, 11, 3, 14]) & df_mod['Team B Seed'].isin([7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([7, 10, 2, 15]) & df_mod['Team B Seed'].isin([6, 11, 3, 14]))
)

round_4_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]) & df_mod['Team B Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15]) & df_mod['Team B Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]))
)

round_5_condition = (
    (df_mod['Team A Region'].isin(['W']) & df_mod['Team B Region'].isin(['X'])) | 
    (df_mod['Team A Region'].isin(['X']) & df_mod['Team B Region'].isin(['W'])) |
    (df_mod['Team A Region'].isin(['Y']) & df_mod['Team B Region'].isin(['Z'])) | 
    (df_mod['Team A Region'].isin(['Z']) & df_mod['Team B Region'].isin(['Y']))
)

round_6_condition = (
    (df_mod['Team A Region'].isin(['W', 'X']) & df_mod['Team B Region'].isin(['Y', 'Z'])) | 
    (df_mod['Team A Region'].isin(['Y', 'Z']) & df_mod['Team B Region'].isin(['W', 'X'])) 
)

round_6_condition

0       False
1       False
2       False
3       False
4       False
        ...  
1507    False
1508    False
1509    False
1510    False
1511     True
Length: 1512, dtype: bool

In [49]:
df_mod['Round'] = -1

df_mod.loc[round_6_condition, 'Round'] = 6

df_mod.loc[round_5_condition, 'Round'] = 5

df_mod.loc[round_4_condition & same_region, 'Round'] = 4

df_mod.loc[round_3_condition & same_region, 'Round'] = 3

df_mod.loc[round_2_condition & same_region, 'Round'] = 2

df_mod.loc[round_1_condition & same_region, 'Round'] = 1

df_mod['Round'].describe()

count    1512.000000
mean        1.904762
std         1.191822
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         6.000000
Name: Round, dtype: float64

In [50]:
assert ((df_mod['Round'] >= 1).all()), 'The round mapping is incorrect'

Get home court advantage

In [51]:
df_mod['Location'] = 0

# conditions: after 2012 but not 2021 (covid stadium), matchup is within first 2 rounds, and team is top 4 seed
df_mod.loc[
    (df_mod['Season'] > 2012) & 
    (df_mod['Season'] != 2021) & 
    (df_mod['Round'] <= 2) & 
    (df_mod['Team A Seed'] <= 4), 
    'Location'
] = 1

df_mod.loc[
    (df_mod['Season'] > 2012) & 
    (df_mod['Season'] != 2021) & 
    (df_mod['Round'] <= 2) & 
    (df_mod['Team B Seed'] <= 4), 
    'Location'
] = -1

df_mod

,Season,Result,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location
0,2012,1,3116,3173,Z,Z,6,11,1,0
1,2012,1,3163,3341,Y,Y,1,16,1,0
2,2012,1,3177,3140,W,W,7,10,1,0
3,2012,1,3211,3353,Y,Y,11,6,1,0
4,2012,1,3243,3343,Y,Y,8,9,1,0
...,...,...,...,...,...,...,...,...,...,...
1507,2024,-1,3425,3163,Z,Z,1,3,4,0
1508,2024,-1,3261,3234,Y,Y,3,1,4,0
1509,2024,-1,3163,3234,Z,Y,3,1,5,0
1510,2024,-1,3301,3376,X,W,3,1,5,0


In [52]:
df_mod['Seed'] = df_mod['Team A Seed'] - df_mod['Team B Seed']

df_mod.drop(columns=['Team A Region', 'Team B Region', 'Team A Seed', 'Team B Seed'], inplace=True)

df_mod

,Season,Result,Team A ID,Team B ID,Round,Location,Seed
0,2012,1,3116,3173,1,0,-5
1,2012,1,3163,3341,1,0,-15
2,2012,1,3177,3140,1,0,-3
3,2012,1,3211,3353,1,0,5
4,2012,1,3243,3343,1,0,-1
...,...,...,...,...,...,...,...
1507,2024,-1,3425,3163,4,0,-2
1508,2024,-1,3261,3234,4,0,2
1509,2024,-1,3163,3234,5,0,2
1510,2024,-1,3301,3376,5,0,2


Get Head-to-Head

In [53]:
df_h2h = pd.read_parquet('../data/preprocessed/womens_h2h/h2h.parquet')

df_h2h

,Season,Team A,Team B,Head to Head,Common Opps
0,2012,Air Force,Alabama,-0.800000,NaN
1,2012,Air Force,Alabama A&M,NaN,0.180669
2,2012,Air Force,Alabama State,NaN,0.052821
3,2012,Air Force,Alcorn State,NaN,0.363068
4,2012,Air Force,American,NaN,-1.447008
...,...,...,...,...,...
739007,2024,Youngstown State,Wichita State,NaN,-0.081696
739008,2024,Youngstown State,Wisconsin,NaN,-0.729834
739009,2024,Youngstown State,Wright State,0.023094,-0.172887
739010,2024,Youngstown State,Wyoming,NaN,-0.684239


In [54]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_h2h['Team A'].unique())

df_match.head(25)

  0%|          | 0/363 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Air Force,air force,100
4,Sam Houston,sam houston,100
5,Saint Peter's,saint peter's,100
6,Saint Mary's (CA),saint mary's (ca),100
7,Saint Louis,saint louis,100
8,Saint Joseph's,saint joseph's,100
9,Saint Francis (PA),saint francis (pa),100


In [55]:
df_h2h.insert(df_h2h.columns.get_loc('Team A'), 'Team A ID', df_h2h['Team A'].map(team_to_spelling).map(spelling_to_id))

df_h2h.insert(df_h2h.columns.get_loc('Team B'), 'Team B ID', df_h2h['Team B'].map(team_to_spelling).map(spelling_to_id))

df_h2h

,Season,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps
0,2012,3102,Air Force,3104,Alabama,-0.800000,NaN
1,2012,3102,Air Force,3105,Alabama A&M,NaN,0.180669
2,2012,3102,Air Force,3106,Alabama State,NaN,0.052821
3,2012,3102,Air Force,3108,Alcorn State,NaN,0.363068
4,2012,3102,Air Force,3110,American,NaN,-1.447008
...,...,...,...,...,...,...,...
739007,2024,3464,Youngstown State,3455,Wichita State,NaN,-0.081696
739008,2024,3464,Youngstown State,3458,Wisconsin,NaN,-0.729834
739009,2024,3464,Youngstown State,3460,Wright State,0.023094,-0.172887
739010,2024,3464,Youngstown State,3461,Wyoming,NaN,-0.684239


In [56]:
df_mod = pd.merge(
    df_mod,
    df_h2h[['Season', 'Team A ID', 'Team B ID', 'Head to Head', 'Common Opps']],
    how='left',
    on=['Season', 'Team A ID', 'Team B ID'],
)

df_mod

,Season,Result,Team A ID,Team B ID,Round,Location,Seed,Head to Head,Common Opps
0,2012,1,3116,3173,1,0,-5,NaN,-0.133288
1,2012,1,3163,3341,1,0,-15,NaN,1.433520
2,2012,1,3177,3140,1,0,-3,NaN,-0.333089
3,2012,1,3211,3353,1,0,5,NaN,NaN
4,2012,1,3243,3343,1,0,-1,NaN,-0.653105
...,...,...,...,...,...,...,...,...,...
1507,2024,-1,3425,3163,4,0,-2,NaN,0.481418
1508,2024,-1,3261,3234,4,0,2,NaN,0.228265
1509,2024,-1,3163,3234,5,0,2,NaN,0.071684
1510,2024,-1,3301,3376,5,0,2,NaN,-0.379131


Get team names

In [57]:
df_teams = pd.read_csv(r'..\data\unprocessed\kaggle\WTeams.csv')

df_teams

,TeamID,TeamName
0,3101,Abilene Chr
1,3102,Air Force
2,3103,Akron
3,3104,Alabama
4,3105,Alabama A&M
...,...,...
373,3476,Stonehill
374,3477,East Texas A&M
375,3478,Le Moyne
376,3479,Mercyhurst


In [58]:
id_to_team = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

df_mod.insert(df_mod.columns.get_loc('Team A ID') + 1, 'Team A', df_mod['Team A ID'].map(id_to_team))
df_mod.insert(df_mod.columns.get_loc('Team B ID') + 1, 'Team B', df_mod['Team B ID'].map(id_to_team))

df_mod

,Season,Result,Team A ID,Team A,Team B ID,Team B,Round,Location,Seed,Head to Head,Common Opps
0,2012,1,3116,Arkansas,3173,Dayton,1,0,-5,NaN,-0.133288
1,2012,1,3163,Connecticut,3341,Prairie View,1,0,-15,NaN,1.433520
2,2012,1,3177,DePaul,3140,BYU,1,0,-3,NaN,-0.333089
3,2012,1,3211,Gonzaga,3353,Rutgers,1,0,5,NaN,NaN
4,2012,1,3243,Kansas St,3343,Princeton,1,0,-1,NaN,-0.653105
...,...,...,...,...,...,...,...,...,...,...,...
1507,2024,-1,3425,USC,3163,Connecticut,4,0,-2,NaN,0.481418
1508,2024,-1,3261,LSU,3234,Iowa,4,0,2,NaN,0.228265
1509,2024,-1,3163,Connecticut,3234,Iowa,5,0,2,NaN,0.071684
1510,2024,-1,3301,NC State,3376,South Carolina,5,0,2,NaN,-0.379131


Map features

In [59]:
team_a_features = pd.merge(
    df_mod[['Season', 'Team A ID']],
    df.drop(columns=['Team']),
    how='left',
    left_on=['Season', 'Team A ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team A ID', 'TeamID'])

team_b_features = pd.merge(
    df_mod[['Season', 'Team B ID']],
    df.drop(columns=['Team']),
    how='left',
    left_on=['Season', 'Team B ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team B ID', 'TeamID'])

df_features = team_a_features - team_b_features

# df_features['Team A ADJOE Team B ADJDE'] = team_a_features['ADJOE'] + team_b_features['ADJDE']
# df_features['Team B ADJOE Team A ADJDE'] = team_b_features['ADJOE'] + team_a_features['ADJDE']

# df_features['Team A Offense Team B Defense'] = team_a_features['Adjusted Offense'] + team_b_features['Adjusted Defense']
# df_features['Team B Offense Team A Defense'] = team_b_features['Adjusted Offense'] + team_a_features['Adjusted Defense']

df_features['Team A Efficiency Margin'] = team_a_features['Efficiency Margin']
df_features['Team B Efficiency Margin'] = team_b_features['Efficiency Margin']

df_features

,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating,Team A Efficiency Margin,Team B Efficiency Margin
0,-1.0,-0.750000,-0.058519,-0.055132,-0.046267,0.030005,0.005625,-0.024380,-8.312479,-1.395719,-0.848218,0.242118,0.212113
1,4.0,2.750000,0.599396,0.587873,5.533340,0.683283,0.335346,-0.347937,1.164885,30.158018,30.046412,0.612579,-0.070705
2,3.0,1.000000,0.116604,0.152977,0.942105,0.021541,0.069822,0.048281,2.527211,-3.043846,-2.413117,0.243769,0.222228
3,2.0,1.000000,0.046283,-0.025164,-0.565226,-0.048145,0.054367,0.102513,4.992007,3.506069,2.932589,0.216886,0.265031
4,0.0,-0.250000,-0.023129,0.142977,0.166964,-0.066541,-0.049703,0.016838,-6.785140,-5.096065,-3.479339,0.203945,0.270486
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1507,-2.0,-4.333333,-0.157742,-0.286640,-0.166886,-0.130033,-0.062637,0.067396,-2.007100,-0.349031,-0.542839,0.373978,0.504011
1508,1.0,-0.666667,0.033278,-0.057085,-0.519727,-0.045249,-0.100185,-0.054936,-0.004142,-1.030708,-0.630969,0.402188,0.447437
1509,-3.0,1.000000,0.019989,0.140849,0.181251,0.056574,-0.058381,-0.114955,-4.357594,0.057720,0.323102,0.504011,0.447437
1510,-4.0,-3.000000,-0.272947,-0.153657,-2.108665,-0.186511,-0.120974,0.065538,-2.295430,-12.321040,-11.483347,0.354143,0.540655


In [60]:
df_mod[df_features.columns] = df_features

df_mod

,Season,Result,Team A ID,Team A,Team B ID,Team B,Round,Location,Seed,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating,Team A Efficiency Margin,Team B Efficiency Margin
0,2012,1,3116,Arkansas,3173,Dayton,1,0,-5,NaN,-0.133288,-1.0,-0.750000,-0.058519,-0.055132,-0.046267,0.030005,0.005625,-0.024380,-8.312479,-1.395719,-0.848218,0.242118,0.212113
1,2012,1,3163,Connecticut,3341,Prairie View,1,0,-15,NaN,1.433520,4.0,2.750000,0.599396,0.587873,5.533340,0.683283,0.335346,-0.347937,1.164885,30.158018,30.046412,0.612579,-0.070705
2,2012,1,3177,DePaul,3140,BYU,1,0,-3,NaN,-0.333089,3.0,1.000000,0.116604,0.152977,0.942105,0.021541,0.069822,0.048281,2.527211,-3.043846,-2.413117,0.243769,0.222228
3,2012,1,3211,Gonzaga,3353,Rutgers,1,0,5,NaN,NaN,2.0,1.000000,0.046283,-0.025164,-0.565226,-0.048145,0.054367,0.102513,4.992007,3.506069,2.932589,0.216886,0.265031
4,2012,1,3243,Kansas St,3343,Princeton,1,0,-1,NaN,-0.653105,0.0,-0.250000,-0.023129,0.142977,0.166964,-0.066541,-0.049703,0.016838,-6.785140,-5.096065,-3.479339,0.203945,0.270486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1507,2024,-1,3425,USC,3163,Connecticut,4,0,-2,NaN,0.481418,-2.0,-4.333333,-0.157742,-0.286640,-0.166886,-0.130033,-0.062637,0.067396,-2.007100,-0.349031,-0.542839,0.373978,0.504011
1508,2024,-1,3261,LSU,3234,Iowa,4,0,2,NaN,0.228265,1.0,-0.666667,0.033278,-0.057085,-0.519727,-0.045249,-0.100185,-0.054936,-0.004142,-1.030708,-0.630969,0.402188,0.447437
1509,2024,-1,3163,Connecticut,3234,Iowa,5,0,2,NaN,0.071684,-3.0,1.000000,0.019989,0.140849,0.181251,0.056574,-0.058381,-0.114955,-4.357594,0.057720,0.323102,0.504011,0.447437
1510,2024,-1,3301,NC State,3376,South Carolina,5,0,2,NaN,-0.379131,-4.0,-3.000000,-0.272947,-0.153657,-2.108665,-0.186511,-0.120974,0.065538,-2.295430,-12.321040,-11.483347,0.354143,0.540655


Try artificial rating adjustment

In [61]:
# df_mod['Result'].corr(df_mod['BARTHAG']), df_mod['Result'].corr(df_mod['Rating']), df_mod['Result'].corr(df_mod['OS Rating'])

In [62]:
# df_temp = df_mod.copy()

# for i in range(2, 7):
#     df_temp.loc[df_temp['Round'] >= i, ['BARTHAG', 'Rating', 'OS Rating']] = df_temp.loc[df_temp['Round'] >= i, ['BARTHAG', 'Rating', 'OS Rating']]*0.95

# df_temp['Result'].corr(df_temp['BARTHAG']), df_temp['Result'].corr(df_temp['Rating']), df_temp['Result'].corr(df_temp['OS Rating'])

In [63]:
# for i in range(2, 7):
#     df_mod.loc[df_mod['Round'] >= i, ['BARTHAG', 'Rating', 'OS Rating']] = df_mod.loc[df_mod['Round'] >= i, ['BARTHAG', 'Rating', 'OS Rating']]*0.95

# df_mod

In [64]:
df_mod.to_parquet('../data/preprocessed/womens_model_data/model_data.parquet')

'Done'

'Done'